# Yönetmelik Asistanı · Colab çalıştırıcı

Bu notebook yalnızca ince bir çalıştırıcıdır (plan 12.2); kodun tamamı depodadır.
VS Code'da Google Colab uzantısıyla açın ve çalışma zamanı olarak **T4 GPU** seçin.

**Faz 0 hedefi:** ortamı kurmak ve `smoke` sağlık kontrolünü geçmek.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1 · GPU kontrolü

In [1]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

name, memory.used [MiB], memory.total [MiB]
Tesla T4, 0 MiB, 15360 MiB


## 2 · Drive ve önbellek dizinleri

İndeks ve veriler her zaman Drive'da tutulur (oturum düşerse kaybolmasın diye).
Model dosyalarını Drive'a yazmak isteğe bağlıdır: ücretsiz Drive kotası 15 GB'tır ve
Qwen3.5-9B (~6,6 GB) ile BGE-M3 ve reranker (~4,5 GB) birlikte bunun büyük kısmını doldurur.
Varsayılan olarak modeller her oturumda yeniden indirilir.

In [3]:
import os
from google.colab import drive

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/yonetmelik-asistani'
MODELLERI_DRIVEDA_TUT = False  # True: indirme süresi kısalır, Drive kotası dolar

os.environ['YA_DATA_DIR'] = f'{BASE}/data'
os.makedirs(os.environ['YA_DATA_DIR'], exist_ok=True)
if MODELLERI_DRIVEDA_TUT:
    os.environ['HF_HOME'] = f'{BASE}/hf-cache'
    os.environ['OLLAMA_MODELS'] = f'{BASE}/ollama-models'
print({k: os.environ.get(k) for k in ('YA_DATA_DIR', 'HF_HOME', 'OLLAMA_MODELS')})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'YA_DATA_DIR': '/content/drive/MyDrive/yonetmelik-asistani/data', 'HF_HOME': None, 'OLLAMA_MODELS': None}


## 3 · Depoyu al ve paketi kur

`REPO_URL` değerini kendi GitHub deponuzla değiştirin. Depo özelse, Colab'ın
*Secrets* bölümüne `GITHUB_TOKEN` ekleyin.

In [4]:
REPO_URL = 'https://github.com/Cnrygt11/yonetmelik-asistani.git'
REPO_DIR = '/content/yonetmelik-asistani'

try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if token:
        REPO_URL = REPO_URL.replace('https://', f'https://{token}@')
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%pip install -q -e {REPO_DIR}

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for yonetmelik-asistani (pyproject.toml) ... done


## 4 · Ollama'yı kur, başlat ve modeli indir

In [5]:
MODEL = 'qwen3.5:9b'  # yedek: 'gemma4:e4b'
!bash {REPO_DIR}/scripts/colab_setup.sh {MODEL}

>> Ollama kuruluyor
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
>> Ollama sunucusu başlatılıyor (log: /content/ollama.log)
>> Model indiriliyor: qwen3.5:9b

NAME          ID              SIZE      MODIFIED               
qwen3.5:9b    56671c2ab938    6.6 GB    Less than a second ago    


## 5 · Sağlık kontrolü

Beş soru çalışır: dördü Türkçe akıcılık ve hız, biri boş bağlamda ret davranışı.
Rapor oturum kapanınca kaybolmasın diye Drive'a yazılır. Faz 0 çıkışında bu raporu
depodaki `eval/reports/` klasörüne ekleyin.

In [6]:
from datetime import datetime
os.makedirs(f'{BASE}/reports', exist_ok=True)
REPORT = f"{BASE}/reports/smoke_{datetime.now():%Y%m%d_%H%M%S}.json"
!cd {REPO_DIR} && python -m yonetmelik_asistani.smoke --model {MODEL} --report {REPORT}
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

GPU      : Tesla T4, 0 MiB, 15360 MiB
Sunucu   : ollama @ http://localhost:11434
Model    : qwen3.5:9b (num_ctx=8192, think=False)

[S1] 127.6 sn · 31.7 token/sn
     Kayıt dondurmak, bir üniversite öğrencisinin eğitimine ara verip resmi olarak kaydını askıya alması anlamına gelir. Bu süreçte öğrenci okulun akademik takviminden çıkar ancak mezuniyet hakkı ve kaydı korunduğu için tekrar derslere başlayıp eğitimini tamamlayabilir.

[S2]   8.9 sn · 31.1 token/sn
     Genel olarak **bütünleme sınavı** ve **mazeret sınavı** arasındaki temel fark, öğrencinin eksikliği gösterdiği sınavın **nedeniyle ilgilidir**:

*   **Bütünleme Sınavı:** Öğrencinin bir sınava girmemesinin sebebi **kendi isteği veya ihmalidir** (örneğin; hastalık, mazeret belgesi yoksa). Bu durumda 

[S3]   4.9 sn · 30.5 token/sn
     Çift ana dal sistemi, öğrencilerin hem genel kültürü hem de belirli bir alanda uzmanlaşmasını sağlamak amacıyla iki farklı alanı dengeli şekilde geliştirerek esnek bir eğitim imkanı sunar. Yan d